# Coffee17 WR-HBP + GCE-LS V1

Strict-deterministic matched experiment: **WR-HBP + CE** vs **WR-HBP + top-2 GCE-LS**.
Only the classification loss changes. Outer test is not accessed.


In [ ]:
NOTEBOOK_BUILD = "WR-HBP-GCE-V1"
SCIENTIFIC_CODE_COMMIT = "8e03b76715f6e64fa8cdef412cf71adf0503d2d5"

import os
# Must be set before CUDA/PyTorch initialization.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import importlib, json, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-wr-hbp-gce-code"
PROJECT = WORK / "coffee17-wr-hbp-gce-project"

assert INPUT.is_dir() and WORK.is_dir(), "Notebook ini harus dijalankan di Kaggle."

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True, env=os.environ.copy())
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=os.environ.copy(),
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

prior_projects = sorted(
    p for p in INPUT.rglob("coffee17-wr-hbp-gce-project")
    if p.is_dir()
)
if len(prior_projects) > 1:
    raise RuntimeError("Lebih dari satu prior WR-HBP-GCE project ditemukan.")
if prior_projects:
    print("MERGE PRIOR PROJECT:", prior_projects[0])
    if PROJECT.exists():
        shutil.rmtree(PROJECT)
    shutil.copytree(prior_projects[0], PROJECT)

PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "logs").mkdir(parents=True, exist_ok=True)
(PROJECT / "analysis").mkdir(parents=True, exist_ok=True)

if REPO.exists():
    shutil.rmtree(REPO)
run([
    "git", "clone", "--quiet", "--no-checkout",
    "https://github.com/ediprin/coffee-bean-classification.git", REPO
])
run(["git", "-C", REPO, "checkout", "--quiet", "--detach", SCIENTIFIC_CODE_COMMIT])

requirements = REPO / "requirements/preprocessing-study.txt"
run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU sebelum Run All.")
print("NOTEBOOK_BUILD:", NOTEBOOK_BUILD)
print("SCIENTIFIC_CODE_COMMIT:", SCIENTIFIC_CODE_COMMIT)
print("GPU:", torch.cuda.get_device_name(0))

# Fail before expensive folds if GCE math or deterministic WR path is broken.
run([
    sys.executable, "-m", "pytest", "-q",
    "tests/engine/test_gradient_boosting_ce.py",
    "tests/modeling/test_wavelet_residual_hbp.py",
], cwd=REPO)

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import (
    materialize_preprocessing_development,
)
from bilinear_lmmd.core.config import load_config
from bilinear_lmmd.engine.physical_logit_residual_wr_hbp import configure_strict_determinism
from bilinear_lmmd.engine.wr_hbp_gce import preflight_matched_initialization

CONFIG = REPO / "configs/wr_hbp_gce/WR_HBP_GCE_V1.yaml"
OUT = PROJECT / "experiments/coffee17-wr-hbp-gce-v1"
OUT.mkdir(parents=True, exist_ok=True)

cfg = load_config(CONFIG)
det = configure_strict_determinism(42)
preflight = preflight_matched_initialization(cfg)
print("\n=== STRICT DETERMINISM ===")
print(json.dumps(det, indent=2))
print("\n=== MATCHED INITIALIZATION ===")
print(json.dumps(preflight, indent=2))

by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        "Input harus Coffee17 original: "
        f"expected 979 images / 17 classes, observed {raw_count}/{len(by_class)}."
    )

ARCHIVE = WORK / "coffee17_wr_hbp_gce_original.zip"
PROV_LOCAL = WORK / "coffee17_wr_hbp_gce_provenance"
CANONICAL = WORK / "coffee17_wr_hbp_gce_original_v1"
FOLDS_LOCAL = WORK / "coffee17_wr_hbp_gce_folds"

for path in (PROV_LOCAL, CANONICAL, FOLDS_LOCAL):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE, PROV_LOCAL, canonical_root=CANONICAL
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL / "coffee17_provenance.json",
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

print("\n=== 5-FOLD WR-HBP CE VS GCE-LS ===")
for fold in range(1, 6):
    print(f"\n================ FOLD {fold}/5 ================", flush=True)
    DEV = WORK / f"coffee17_wr_hbp_gce_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)

    materialize_preprocessing_development(
        CANONICAL,
        FOLDS_LOCAL / "clean_manifest.json",
        FOLDS_LOCAL / "fold_manifest.json",
        DEV,
        fold=fold,
    )

    run([
        sys.executable, "-u", "-m",
        "bilinear_lmmd.experiments.run_wr_hbp_gce_matched",
        "--config", CONFIG,
        "--fold", str(fold),
        "--data-root", DEV,
        "--output-root", OUT,
        "--required-commit", SCIENTIFIC_CODE_COMMIT,
        "--device", "cuda:0",
        "--resume",
        "--authorize-training",
    ], cwd=REPO, log_path=PROJECT / "logs" / f"WR_HBP_GCE_fold{fold}.log")

    result = json.loads(
        (OUT / f"fold_{fold}" / "seed42" / "pair_result.json").read_text()
    )
    d = result["DELTA_GCE_MINUS_CE"]
    print(
        f"FOLD {fold} | "
        f"CE Macro={result['WR_HBP_CE']['macro_f1']:.4f} | "
        f"GCE Macro={result['WR_HBP_GCE']['macro_f1']:.4f} | "
        f"Delta Macro={d['macro_f1']:+.4f} | "
        f"Delta Hard={d['hard_class_f1']:+.4f} | "
        f"Delta Worst={d['worst_class_f1']:+.4f} | "
        f"TargetPairDelta={result['targeted_confusion_delta']['TOTAL']:+d}"
    )

    shutil.rmtree(DEV, ignore_errors=True)
    torch.cuda.empty_cache()

SUMMARY = PROJECT / "analysis" / "wr_hbp_gce_summary.json"
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_wr_hbp_gce_summary",
    "--output-root", OUT,
    "--output", SUMMARY,
], cwd=REPO)

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
agg = summary["aggregate"]
d = agg["DELTA_GCE_MINUS_CE"]
print("\n=== KEY RESULT ===")
print("CE  Macro-F1 :", f"{agg['WR_HBP_CE']['macro_f1']['mean']:.4%}")
print("GCE Macro-F1 :", f"{agg['WR_HBP_GCE']['macro_f1']['mean']:.4%}")
print("Delta Macro  :", f"{d['macro_f1']['mean']:+.4%}")
print("CE  Hard-F1  :", f"{agg['WR_HBP_CE']['hard_class_f1']['mean']:.4%}")
print("GCE Hard-F1  :", f"{agg['WR_HBP_GCE']['hard_class_f1']['mean']:.4%}")
print("Delta Hard   :", f"{d['hard_class_f1']['mean']:+.4%}")
print("CE  Worst-F1 :", f"{agg['WR_HBP_CE']['worst_class_f1']['mean']:.4%}")
print("GCE Worst-F1 :", f"{agg['WR_HBP_GCE']['worst_class_f1']['mean']:.4%}")
print("Delta Worst  :", f"{d['worst_class_f1']['mean']:+.4%}")
print("Hard positive folds:", summary["positive_delta_folds"]["hard_class_f1"], "/ 5")
print("Target pair total delta:", summary["targeted_confusion_delta_total"]["TOTAL"])
print("SCREENING GATE:", summary["screening_gate"]["decision"])

PKG = WORK / "wr-hbp-gce-analysis-package"
shutil.rmtree(PKG, ignore_errors=True)
PKG.mkdir(parents=True)
shutil.copytree(PROJECT / "analysis", PKG / "analysis")

for fold in range(1, 6):
    pair_src = OUT / f"fold_{fold}" / "seed42"
    pair_dst = PKG / f"fold_{fold}"
    pair_dst.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pair_src / "pair_result.json", pair_dst / "pair_result.json")
    for arm in ("WR_HBP_CE", "WR_HBP_GCE"):
        src = pair_src / arm
        dst = pair_dst / arm
        dst.mkdir(parents=True, exist_ok=True)
        for name in ("run_contract.json", "history.json", "run_config.yaml"):
            p = src / name
            if p.is_file():
                shutil.copy2(p, dst / name)
        if (src / "validation").is_dir():
            shutil.copytree(src / "validation", dst / "validation")

shutil.copy2(CONFIG, PKG / "WR_HBP_GCE_V1.yaml")
shutil.copy2(REPO / "docs/protocols/WR_HBP_GCE_V1.md", PKG / "WR_HBP_GCE_V1.md")

zip_path = Path(shutil.make_archive(
    str(WORK / "wr-hbp-gce-analysis-package"),
    "zip",
    root_dir=PKG,
))
print("\nREADY:", zip_path)
print("SIZE:", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")
print("Outer test was NOT accessed.")

for path in (REPO, PROV_LOCAL, CANONICAL, FOLDS_LOCAL, PKG):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
